# Compare repeated workflow runs across E01, E02, and E03

Set PILOT_WORKFLOW_RUNS to a JSONL file with case, environment, round, and
run_id for each accepted sample. Include five clean samples per combination. Use the usual Airflow URL and token variables.
A separate endpoint for a workflow can use AIRFLOW_LOCAL_<CASE>_URL and
AIRFLOW_LOCAL_<CASE>_TOKEN.

[English guide](../../docs/operations-en.md) · [한국어 가이드](../../docs/operations-ko.md)

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path
from time import sleep

import httpx
import pandas as pd
from IPython.display import display

from pilot_kit import airflow_api, workflows

environments = {"local": "E01", "docker": "E02", "kubernetes": "E03"}
titles = {
    workflows.Case.LOOP: "Reconciliation loop",
    workflows.Case.TYPESCRIPT: "TypeScript settlement",
    workflows.Case.MIXED: "Mixed-language chain",
}
records = [
    json.loads(line)
    for line in Path(os.environ["PILOT_WORKFLOW_RUNS"]).read_text().splitlines()
    if line.strip()
]
expected = {
    (case, target, round_no)
    for case in workflows.Case
    for target in environments
    for round_no in range(1, 6)
}
planned = {(row["case"], row["environment"], row["round"]) for row in records}
if planned != expected or len(records) != len(expected):
    raise ValueError("Provide one run for every workflow, environment, and round.")
if len({row["run_id"] for row in records}) != len(records):
    raise ValueError("Provide distinct run IDs.")

In [ ]:
def workflow_client(target, case):
    prefix = f"AIRFLOW_{target.upper()}_{case.upper()}"
    address = os.environ.get(f"{prefix}_URL")
    token = os.environ.get(f"{prefix}_TOKEN")
    if address or token:
        if not address or not token:
            raise ValueError(f"Set both {prefix}_URL and {prefix}_TOKEN.")
        return airflow_api.client_for(address, token)
    return airflow_api.connect(target)


rows = []
for case in workflows.Case:
    for target, environment in environments.items():
        with workflow_client(target, case) as client:
            client.timeout = httpx.Timeout(45.0)
            for row in records:
                if (row["case"], row["environment"]) != (case, target):
                    continue
                run_id = row["run_id"]
                for attempt in range(3):
                    try:
                        run = airflow_api.get_run(
                            client, run_id, dag_id=workflows.DAG_IDS[case], timeout=45
                        )
                        checked = workflows.verify(client, case, run)
                        break
                    except httpx.TransportError:
                        if attempt == 2:
                            raise
                        sleep(3)
                clean = checked["matches"] and all(
                    task["try_number"] == 1
                    for task in checked["task_states"]
                    if task["state"] == "success"
                )
                if not clean:
                    raise ValueError(f"{run_id} failed verification or included a task retry.")
                start = datetime.fromisoformat(run["start_date"])
                end = datetime.fromisoformat(run["end_date"])
                rows.append(
                    {
                        "case": case.value,
                        "environment": environment,
                        "round": row["round"],
                        "run_id": run_id,
                        "state": checked["state"],
                        "matches": checked["matches"],
                        "duration_seconds": (end - start).total_seconds(),
                        "start_date": start.isoformat(),
                        "end_date": end.isoformat(),
                    }
                )

runs = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", None):
    display(runs)
print(f"Verified runs: {runs['matches'].sum()}/{len(runs)}")

In [ ]:
caption = (
    "Infrastructure differs: E01 and E02 share a local Docker host; E03 uses Kubernetes nodes.\n"
    "Each line shows five accepted samples. Infrastructure and run schedules affect these timings."
)
for case, title in titles.items():
    observations = runs.loc[runs["case"].eq(case)]
    summary = (
        observations.groupby("environment", sort=False)["duration_seconds"]
        .agg(["count", "median", "mean", "min", "max"])
        .round(3)
    )
    display(summary)
    points = observations.pivot(index="round", columns="environment", values="duration_seconds")
    ax = points.plot(
        style="-o",
        figsize=(8, 4.6),
        xlabel="Accepted sample",
        ylabel="DAG duration (s)",
        xticks=range(1, 6),
        title=f"{title} duration by accepted sample",
    )
    ax.set_ylim(bottom=0)
    figure = ax.get_figure()
    figure.subplots_adjust(bottom=0.23)
    figure.text(0.02, 0.04, caption, fontsize=9)